In [0]:
# Databricks notebook source
# ============================================================================
# 02. SILVER LAYER - UN-NESTING & BML PARSING
# Objective: Explode 20k line-item arrays, parse maps, and clean schemas.
# ============================================================================

from pyspark.sql.functions import col, explode, to_timestamp, expr

BRONZE_TABLE = "default.bronze_cpq_quotes"
SILVER_TABLE = "default.silver_cpq_quote_items"

# COMMAND ----------
df_bronze = spark.table(BRONZE_TABLE)

# 1. Explode array of line items into individual row records
df_exploded = df_bronze.select(
    col("transactionId").alias("quote_id"),
    col("customerName").alias("customer_name"),
    to_timestamp(col("quoteDate")).alias("quote_date"),
    col("currency"),
    explode(col("lineItems")).alias("item")
)
# df_exploded.show()
# 2. Extract fields & Parse BML attribute string into Spark Map
df_silver = df_exploded.select(
    "quote_id",
    "customer_name",
    "quote_date",
    "currency",
    col("item.sequence_number").alias("item_seq"),
    col("item.part_number").alias("part_number"),
    col("item.product_category").alias("product_category"),
    col("item.quantity").alias("quantity"),
    col("item.unit_cost").alias("unit_cost"),
    col("item.list_price").alias("list_price"),
    col("item.net_price").alias("net_price"),
    # Convert "key1=val1;key2=val2" string into native Map
    expr("str_to_map(item.bml_attributes_str, ';', '=')").alias("bml_map")
).select(
    "*",
    col("bml_map")["region"].alias("region"),
    col("bml_map")["approval_required"].alias("approval_required")
).drop("bml_map")
# df_silver.describe()
# COMMAND ----------
# 3. Repartition for 20k Volume & Write to Silver Delta Table
df_silver.repartition(8) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_TABLE)

print(f" Successful Silver Transformation: {df_silver.count():,} line items written to {SILVER_TABLE}")